# Step 7 — Emulator Inference Time Benchmark (NTS/mcmc-trained model)

Measures the emulator's own forward-pass cost, isolated from data loading, plotting, and file I/O (unlike timing the whole `step5_test2.py` script, which is dominated by loading 10 checkpoints and rendering plots). Produces a batch-size sweep of wall-clock timings, a `speedup = t_ABM / t_emulator` ratio against the MCMC simulation cost already reported in `experiments/Regression/Results_Combined.ipynb`, and a break-even query count `N*` that accounts for the one-off training cost.

In [7]:
import sys
import json
import time
import types
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from scipy import stats
from step0_model import create_hybrid_mlp_model
from utils import get_device, PARAM_MINS, PARAM_MAXS

# Resolve repo root regardless of whether the notebook's cwd is the repo
# root (e.g. papermill/CI) or this scripts/ folder (e.g. VS Code Jupyter).
SCRIPT_DIR = Path.cwd()
if (SCRIPT_DIR / "step0_model.py").exists():
    REPO_ROOT = SCRIPT_DIR.parents[2]   # scripts -> mcmc-sampling -> experiments -> root
else:
    REPO_ROOT = SCRIPT_DIR
    SCRIPT_DIR = REPO_ROOT / "experiments" / "mcmc-sampling" / "scripts"
sys.path.append(str(SCRIPT_DIR))

#I/O

MODELS_DIR  = REPO_ROOT / "experiments/mcmc-sampling/out/trained-models"
RESULTS_DIR = REPO_ROOT / "experiments/mcmc-sampling/out/results/emulator_time"


N_TIMEPOINTS = 250
BATCH_SIZES  = [1, 35, 64, 1024, 8192]   # 35 = training minibatch size, for reference
N_REPS = 1000   # timed forward passes per batch size
N_WARMUP = 50     # untimed calls to prime Python caches

device = get_device()


Using CPU


## Load one trained replicate

Timing depends only on the frozen architecture and weights, not on which replicate — replicate 1 is representative. Loading happens once, outside the timed region, exactly like a deployed emulator would load weights once and then answer many queries.

In [8]:
def load_replicate_model(model_path: Path, device: torch.device):
    ckpt = torch.load(model_path, map_location=device, weights_only=False)
    config = ckpt.get('config', {
        'n_params': 3, 'n_fourier': 64, 'sigma': 1.0,
        'fusion_hidden': 128, 'latent_dim': 64, 'decoder_hidden': 64,
        'dropout': 0.3, 'n_knots': 8,
        'n_timepoints': N_TIMEPOINTS, 'total_population': 100_000,
    })
    state = ckpt['model_state_dict']
    state.pop('temporal_decoder.t_grid', None)  # remove legacy buffer if present
    model = create_hybrid_mlp_model(config)
    model.load_state_dict(state, strict=True)
    model.to(device).eval()
    return model, config


model_path = sorted(
    MODELS_DIR.glob("best_balanced_mlp_model_*.pt"),
    key=lambda p: int(p.stem.split('_')[-1])
)[0]
model, config = load_replicate_model(model_path, device)
print(f"Loaded {model_path.name} on {device}")

Loaded best_balanced_mlp_model_1.pt on cpu


## Benchmark methodology

Random `(\u03c4, \u03b3, \u03c1)` inputs are generated directly on `device` — the real test-set pickle is never loaded, since only the model's own compute cost is of interest here, not `DataLoader`/pickle I/O. For each batch size: `N_WARMUP` untimed calls prime CUDA/cuDNN and Python's caches, then `N_REPS` timed calls are each wrapped with `torch.cuda.synchronize()` (a no-op on CPU) so async GPU kernels aren't under-counted.

In [9]:
def make_batch(batch_size: int, device: torch.device):
    params_norm = torch.rand(batch_size, 3, device=device)
    rho_raw = torch.empty(batch_size, device=device).uniform_(
        float(PARAM_MINS[2]), float(PARAM_MAXS[2])
    )
    return types.SimpleNamespace(params_norm=params_norm, rho_raw=rho_raw)


def time_forward_pass(model, batch_size, device, n_reps=N_REPS, n_warmup=N_WARMUP):
    batch = make_batch(batch_size, device)
    sync = torch.cuda.synchronize if device.type == 'cuda' else (lambda: None)

    with torch.no_grad():
        for _ in range(n_warmup):
            _ = model(batch, n_timesteps=N_TIMEPOINTS)
        sync()

        times = np.empty(n_reps)
        for i in range(n_reps):
            t0 = time.perf_counter()
            _ = model(batch, n_timesteps=N_TIMEPOINTS)
            sync()
            times[i] = time.perf_counter() - t0

    return times

In [10]:
def summarise(times: np.ndarray, batch_size: int) -> dict:
    """Mean/SD/SEM/95% t-CI over the n_reps timed calls, plus derived rates."""
    n = len(times)
    mean, std = float(times.mean()), float(times.std(ddof=1))
    sem = std / np.sqrt(n)
    ci = stats.t.interval(0.95, df=n - 1, loc=mean, scale=sem)
    return {
        'batch_size'              : batch_size,
        'mean_batch_time_s'       : mean,
        'std_s'                   : std,
        'sem_s'                   : sem,
        'ci95_lower_s'            : float(ci[0]),
        'ci95_upper_s'            : float(ci[1]),
        'mean_per_query_time_s'   : mean / batch_size,
        'throughput_queries_per_s': batch_size / mean,
    }


rows = []
for bs in BATCH_SIZES:
    times = time_forward_pass(model, bs, device)
    rows.append(summarise(times, bs))
    print(f"batch={bs:>5}  "
          f"per-query={rows[-1]['mean_per_query_time_s']*1e3:.4f} ms  "
          f"throughput={rows[-1]['throughput_queries_per_s']:,.0f} queries/s")

timing_df = pd.DataFrame(rows)
timing_df

batch=    1  per-query=0.4496 ms  throughput=2,224 queries/s
batch=   35  per-query=0.0236 ms  throughput=42,382 queries/s
batch=   64  per-query=0.0078 ms  throughput=128,032 queries/s
batch= 1024  per-query=0.0020 ms  throughput=490,141 queries/s
batch= 8192  per-query=0.0013 ms  throughput=782,642 queries/s


,batch_size,mean_batch_time_s,std_s,sem_s,ci95_lower_s,ci95_upper_s,mean_per_query_time_s,throughput_queries_per_s
0,1,0.000450,0.000409,0.000013,0.000424,0.000475,0.000450,2224.037525
1,35,0.000826,0.000341,0.000011,0.000805,0.000847,0.000024,42382.245412
2,64,0.000500,0.000297,0.000009,0.000481,0.000518,0.000008,128032.059412
3,1024,0.002089,0.000591,0.000019,0.002053,0.002126,0.000002,490141.244826
4,8192,0.010467,0.001120,0.000035,0.010398,0.010537,0.000001,782642.248412


## Speed-up vs. the ABM

Reference ABM cost: `simulation_time(per simulation)` for the MCMC/NTS strategy in `experiments/Regression/Results_Combined.ipynb` is **0.00025 min = 0.015 s** per `EoN.fast_SIR` run (single simulation, excludes the one-off network build already amortised in that notebook's `Time_all_sim` column). This is the fair single-query comparison point (batch = 1); the larger batch sizes below show the additional speed-up available when the emulator is queried in bulk (sensitivity analysis, calibration, policy grid search), which the ABM cannot exploit in the same way.

In [11]:
ABM_TIME_PER_SIM_S = 0.00025 * 60   # MCMC/NTS strategy, from Results_Combined.ipynb

timing_df['speedup_vs_abm'] = ABM_TIME_PER_SIM_S / timing_df['mean_per_query_time_s']
print(timing_df[['batch_size', 'mean_per_query_time_s',
                  'throughput_queries_per_s', 'speedup_vs_abm']]
      .to_string(index=False))

# Break-even query count: training is a one-off cost, ABM cost is paid per query.
# Solve N* where N* x t_ABM == training_time + N* x t_emulator (batch=1, single-query case)
summary = json.load(open(MODELS_DIR / 'replicates_summary.json'))
training_time_s = summary['statistics']['training_time_minutes']['mean'] * 60

t_emulator_q1 = timing_df.loc[timing_df.batch_size == 1, 'mean_per_query_time_s'].iloc[0]
n_break_even = training_time_s / (ABM_TIME_PER_SIM_S - t_emulator_q1)

print(f"\nMean training time (k=10 replicates): {training_time_s:.1f} s")
print(f"Break-even query count N*: {n_break_even:,.0f}")
print("  Below N* the ABM is still cheaper overall; "
      "above N* the emulator has paid off its training cost.")

 batch_size  mean_per_query_time_s  throughput_queries_per_s  speedup_vs_abm
          1               0.000450               2224.037525       33.360563
         35               0.000024              42382.245412      635.733681
         64               0.000008             128032.059412     1920.480891
       1024               0.000002             490141.244826     7352.118672
       8192               0.000001             782642.248412    11739.633726

Mean training time (k=10 replicates): 186.1 s
Break-even query count N*: 12,788
  Below N* the ABM is still cheaper overall; above N* the emulator has paid off its training cost.


## Save results

In [ ]:
timing_df.to_csv(RESULTS_DIR / 'emulator_inference_time.csv', index=False)

report = {
    'device'                : str(device),
    'model_path'            : str(model_path),
    'n_reps_per_batch_size' : N_REPS,
    'n_warmup'              : N_WARMUP,
    'abm_time_per_sim_s'    : ABM_TIME_PER_SIM_S,
    'training_time_mean_s'  : training_time_s,
    'break_even_n_queries'  : n_break_even,
    'timings'               : rows,
}
with open(RESULTS_DIR / 'emulator_time_report.json', 'w') as f:
    json.dump(report, f, indent=2)

print(f"Saved: {RESULTS_DIR.resolve()}")

Saved: C:\Users\gmm71\OneDrive - University of Cambridge\Documents\Emulation models\abm-epidemic-emulator\experiments\mcmc-sampling\out\results\emulator_time


: 